<a href="https://colab.research.google.com/github/8458076318/AI-ML-Training-Projects/blob/dev/student_health.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers torch

In [ ]:
!pip install numpy pandas seaborn

# SQL


In [1]:
import pandas as pd
import sqlite3

# Create an in-memory SQLite database connection
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

print("Setup complete. Environment ready.")

# Define a messy, unnormalized dataset (0NF)
# Issues: Non-atomic values (multiple text books), redundant data, mixed concerns.
data_0nf = {
    "Student_ID": [101, 101, 102, 103, 104],
    "Student_Name": ["Alice", "Alice", "Bob", "Charlie", "David"],
    "Course_Code": ["CS101", "MATH201", "CS101", "CS102", "MATH201"],
    "Course_Name": ["Intro to CS", "Calculus I", "Intro to CS", "Data Structures", "Calculus I"],
    "Instructor_ID": ["INS_01", "INS_02", "INS_01", "INS_03", "INS_05"],
    "Instructor_Name": ["Dr. Smith", "Dr. Jones", "Dr. Smith", "Dr. Alan", "Dr. Jones"],
    "Instructor_Office": ["Room 401", "Room 502", "Room 401", "Room 401", "Room 502"],
    "Textbooks_Required": ["Python Basics, Intro to CLI", "Calculus Vol 1", "Python Basics, Intro to CLI", "Algorithms Vol 1", "Calculus Vol 1"],
    "Grade": ["A", "B", "A", "B+", "A-"]
}

df_0nf = pd.DataFrame(data_0nf)
df_0nf.to_sql('Unnormalized_Leasing', conn, index=False, if_exists='replace')

print("\n--- Unnormalized Data (0NF) ---")
df_0nf

# ONF dummy data

# Flattening the Textbooks_Required column to ensure atomicity
df_1nf = df_0nf.assign(Textbooks_Required=df_0nf['Textbooks_Required'].str.split(', ')).explode('Textbooks_Required')

# Save to SQL
df_1nf.to_sql('Table_1NF', conn, index=False, if_exists='replace')

print("--- 1NF Data (Atomic values enforced) ---")
print(f"Row count increased from {len(df_0nf)} to {len(df_1nf)} due to flattening.")
df_1nf

# code for doing first normalization on the table

Setup complete. Environment ready.

--- Unnormalized Data (0NF) ---
--- 1NF Data (Atomic values enforced) ---
Row count increased from 5 to 7 due to flattening.


,Student_ID,Student_Name,Course_Code,Course_Name,Instructor_ID,Instructor_Name,Instructor_Office,Textbooks_Required,Grade
0,101,Alice,CS101,Intro to CS,INS_01,Dr. Smith,Room 401,Python Basics,A
0,101,Alice,CS101,Intro to CS,INS_01,Dr. Smith,Room 401,Intro to CLI,A
1,101,Alice,MATH201,Calculus I,INS_02,Dr. Jones,Room 502,Calculus Vol 1,B
2,102,Bob,CS101,Intro to CS,INS_01,Dr. Smith,Room 401,Python Basics,A
2,102,Bob,CS101,Intro to CS,INS_01,Dr. Smith,Room 401,Intro to CLI,A
3,103,Charlie,CS102,Data Structures,INS_03,Dr. Alan,Room 401,Algorithms Vol 1,B+
4,104,David,MATH201,Calculus I,INS_05,Dr. Jones,Room 502,Calculus Vol 1,A-


In [3]:
import pandas as pd
import sqlite3

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# ─────────────────────────────────────────
# ORIGINAL 0NF DATA
# ─────────────────────────────────────────
data_0nf = {
    "Student_ID": [101, 101, 102, 103, 104],
    "Student_Name": ["Alice", "Alice", "Bob", "Charlie", "David"],
    "Course_Code": ["CS101", "MATH201", "CS101", "CS102", "MATH201"],
    "Course_Name": ["Intro to CS", "Calculus I", "Intro to CS", "Data Structures", "Calculus I"],
    "Instructor_ID": ["INS_01", "INS_02", "INS_01", "INS_03", "INS_05"],
    "Instructor_Name": ["Dr. Smith", "Dr. Jones", "Dr. Smith", "Dr. Alan", "Dr. Jones"],
    "Instructor_Office": ["Room 401", "Room 502", "Room 401", "Room 401", "Room 502"],
    "Textbooks_Required": ["Python Basics, Intro to CLI", "Calculus Vol 1",
                           "Python Basics, Intro to CLI", "Algorithms Vol 1", "Calculus Vol 1"],
    "Grade": ["A", "B", "A", "B+", "A-"]
}

df_0nf = pd.DataFrame(data_0nf)
print("--- Unnormalized Data (0NF) ---")
print(df_0nf)

# ─────────────────────────────────────────
# 1NF — Atomic values + Composite PK
# ─────────────────────────────────────────
df_1nf = (df_0nf
          .assign(Textbooks_Required=df_0nf['Textbooks_Required'].str.split(', '))
          .explode('Textbooks_Required')
          .reset_index(drop=True))

df_1nf.to_sql('Table_1NF', conn, index=False, if_exists='replace')

print("\n--- 1NF Data (Atomic values enforced) ---")
print(f"Row count increased from {len(df_0nf)} to {len(df_1nf)} due to flattening.")
print(f"Primary Key: (Student_ID, Course_Code, Textbooks_Required)")
print(df_1nf)



--- Unnormalized Data (0NF) ---
   Student_ID Student_Name Course_Code      Course_Name Instructor_ID  \
0         101        Alice       CS101      Intro to CS        INS_01   
1         101        Alice     MATH201       Calculus I        INS_02   
2         102          Bob       CS101      Intro to CS        INS_01   
3         103      Charlie       CS102  Data Structures        INS_03   
4         104        David     MATH201       Calculus I        INS_05   

  Instructor_Name Instructor_Office           Textbooks_Required Grade  
0       Dr. Smith          Room 401  Python Basics, Intro to CLI     A  
1       Dr. Jones          Room 502               Calculus Vol 1     B  
2       Dr. Smith          Room 401  Python Basics, Intro to CLI     A  
3        Dr. Alan          Room 401             Algorithms Vol 1    B+  
4       Dr. Jones          Room 502               Calculus Vol 1    A-  

--- 1NF Data (Atomic values enforced) ---
Row count increased from 5 to 7 due to flattenin

In [4]:
# ─────────────────────────────────────────
# 2NF — Remove Partial Dependencies
# Split into 4 tables
# ─────────────────────────────────────────

# Table 1: Students (Student_ID → Student_Name)
df_students = df_1nf[['Student_ID', 'Student_Name']].drop_duplicates().reset_index(drop=True)
df_students.to_sql('Students', conn, index=False, if_exists='replace')

# Table 2: Courses (Course_Code → Course_Name, Instructor_ID)
df_courses = df_1nf[['Course_Code', 'Course_Name', 'Instructor_ID',
                      'Instructor_Name', 'Instructor_Office']].drop_duplicates().reset_index(drop=True)
df_courses.to_sql('Courses', conn, index=False, if_exists='replace')

# Table 3: Course_Textbooks (Course_Code + Textbooks_Required)
df_textbooks = df_1nf[['Course_Code', 'Textbooks_Required']].drop_duplicates().reset_index(drop=True)
df_textbooks.to_sql('Course_Textbooks', conn, index=False, if_exists='replace')

# Table 4: Enrollments (Student_ID + Course_Code → Grade)
df_enrollments = df_1nf[['Student_ID', 'Course_Code', 'Grade']].drop_duplicates().reset_index(drop=True)
df_enrollments.to_sql('Enrollments', conn, index=False, if_exists='replace')

# ─────────────────────────────────────────
# PRINT 2NF TABLES
# ─────────────────────────────────────────
print("\n--- 2NF Tables (Partial Dependencies Removed) ---")

print("\n[1] Students Table  |  PK: Student_ID")
print(df_students)

print("\n[2] Courses Table  |  PK: Course_Code")
print(df_courses)

print("\n[3] Course_Textbooks Table  |  PK: (Course_Code, Textbooks_Required)")
print(df_textbooks)

print("\n[4] Enrollments Table  |  PK: (Student_ID, Course_Code)")
print(df_enrollments)


--- 2NF Tables (Partial Dependencies Removed) ---

[1] Students Table  |  PK: Student_ID
   Student_ID Student_Name
0         101        Alice
1         102          Bob
2         103      Charlie
3         104        David

[2] Courses Table  |  PK: Course_Code
  Course_Code      Course_Name Instructor_ID Instructor_Name Instructor_Office
0       CS101      Intro to CS        INS_01       Dr. Smith          Room 401
1     MATH201       Calculus I        INS_02       Dr. Jones          Room 502
2       CS102  Data Structures        INS_03        Dr. Alan          Room 401
3     MATH201       Calculus I        INS_05       Dr. Jones          Room 502

[3] Course_Textbooks Table  |  PK: (Course_Code, Textbooks_Required)
  Course_Code Textbooks_Required
0       CS101      Python Basics
1       CS101       Intro to CLI
2     MATH201     Calculus Vol 1
3       CS102   Algorithms Vol 1

[4] Enrollments Table  |  PK: (Student_ID, Course_Code)
   Student_ID Course_Code Grade
0         101    

In [6]:
# ─────────────────────────────────────────
# 2NF — Remove Partial Dependencies
# ─────────────────────────────────────────

# Students: Student_ID → Student_Name
df_students_2nf = (df_1nf[['Student_ID', 'Student_Name']]
                   .drop_duplicates().reset_index(drop=True))

# Courses (with instructor): Course_Code → Course_Name, Instructor details
df_courses_2nf = (df_1nf[['Course_Code', 'Course_Name',
                            'Instructor_ID', 'Instructor_Name', 'Instructor_Office']]
                  .drop_duplicates().reset_index(drop=True))

# Course_Textbooks: (Course_Code, Textbooks_Required)
df_textbooks_2nf = (df_1nf[['Course_Code', 'Textbooks_Required']]
                    .drop_duplicates().reset_index(drop=True))

# Enrollments: (Student_ID, Course_Code) → Grade
df_enrollments_2nf = (df_1nf[['Student_ID', 'Course_Code', 'Grade']]
                      .drop_duplicates().reset_index(drop=True))

# Save to SQL
df_students_2nf.to_sql('Students_2NF', conn, index=False, if_exists='replace')
df_courses_2nf.to_sql('Courses_2NF', conn, index=False, if_exists='replace')
df_textbooks_2nf.to_sql('Course_Textbooks_2NF', conn, index=False, if_exists='replace')
df_enrollments_2nf.to_sql('Enrollments_2NF', conn, index=False, if_exists='replace')

print("\n--- 2NF: Partial Dependencies Removed ---")
print("\n[1] Students_2NF  |  PK: Student_ID")
print(df_students_2nf)
print("\n[2] Courses_2NF  |  PK: Course_Code")
print(df_courses_2nf)
print("\n[3] Course_Textbooks_2NF  |  PK: (Course_Code, Textbooks_Required)")
print(df_textbooks_2nf)
print("\n[4] Enrollments_2NF  |  PK: (Student_ID, Course_Code)")
print(df_enrollments_2nf)

# ─────────────────────────────────────────
# 3NF — Remove Transitive Dependencies
# Transitive chain: Course_Code → Instructor_ID → Instructor_Name, Instructor_Office
# ─────────────────────────────────────────

# Extract Instructors into separate table
df_instructors_3nf = (df_courses_2nf[['Instructor_ID', 'Instructor_Name', 'Instructor_Office']]
                      .drop_duplicates().reset_index(drop=True))

# Clean up Courses — keep only Instructor_ID as Foreign Key
df_courses_3nf = (df_courses_2nf[['Course_Code', 'Course_Name', 'Instructor_ID']]
                  .drop_duplicates().reset_index(drop=True))

# Save to SQL
df_instructors_3nf.to_sql('Instructors_3NF', conn, index=False, if_exists='replace')
df_courses_3nf.to_sql('Courses_3NF', conn, index=False, if_exists='replace')

print("\n--- 3NF: Transitive Dependencies Removed ---")
print("\n[Courses_3NF]  |  PK: Course_Code  |  FK: Instructor_ID")
print(df_courses_3nf)
print("\n[Instructors_3NF]  |  PK: Instructor_ID")
print(df_instructors_3nf)


--- 2NF: Partial Dependencies Removed ---

[1] Students_2NF  |  PK: Student_ID
   Student_ID Student_Name
0         101        Alice
1         102          Bob
2         103      Charlie
3         104        David

[2] Courses_2NF  |  PK: Course_Code
  Course_Code      Course_Name Instructor_ID Instructor_Name Instructor_Office
0       CS101      Intro to CS        INS_01       Dr. Smith          Room 401
1     MATH201       Calculus I        INS_02       Dr. Jones          Room 502
2       CS102  Data Structures        INS_03        Dr. Alan          Room 401
3     MATH201       Calculus I        INS_05       Dr. Jones          Room 502

[3] Course_Textbooks_2NF  |  PK: (Course_Code, Textbooks_Required)
  Course_Code Textbooks_Required
0       CS101      Python Basics
1       CS101       Intro to CLI
2     MATH201     Calculus Vol 1
3       CS102   Algorithms Vol 1

[4] Enrollments_2NF  |  PK: (Student_ID, Course_Code)
   Student_ID Course_Code Grade
0         101       CS101     A
1

# Student Health


In [7]:
import pandas as pd
import sqlite3

conn = sqlite3.connect(':memory:')

# Students
pd.DataFrame({
    "Student_ID": [101, 102, 104],
    "Student_Name": ["Alice", "Bob", "David"]
}).to_sql('Students', conn, index=False, if_exists='replace')

# Clinics
pd.DataFrame({
    "Clinic_Name": ["General Medicine", "Sports Med"]
}).to_sql('Clinics', conn, index=False, if_exists='replace')

# Doctors
pd.DataFrame({
    "Doctor_ID": ["DOC_XYZ", "DOC_ABC"],
    "Doctor_Name": ["Dr. Evans", "Dr. Green"],
    "Clinic_Name": ["General Medicine", "Sports Med"]
}).to_sql('Doctors', conn, index=False, if_exists='replace')

# Visits
pd.DataFrame({
    "Visit_ID": [5001, 5002, 5003],
    "Student_ID": [101, 102, 104],
    "Doctor_ID": ["DOC_XYZ", "DOC_ABC", "DOC_XYZ"]
}).to_sql('Visits', conn, index=False, if_exists='replace')

# Prescriptions
pd.DataFrame({
    "Visit_ID": [5001, 5001, 5002, 5003],
    "Prescription": ["Amoxicillin", "Ibuprofen", "Bandages", "Vitamin D"]
}).to_sql('Prescriptions', conn, index=False, if_exists='replace')

4

In [8]:
# eytraining/Day1/task2.py
import pandas as pd
import sqlite3

conn_challenge = sqlite3.connect(':memory:')

challenge_data = {
    "Visit_ID": [5001, 5001, 5002, 5003],
    "Student_ID": [101, 101, 102, 104],
    "Student_Name": ["Alice", "Alice", "Bob", "David"],
    "Doctor_ID": ["DOC_XYZ", "DOC_XYZ", "DOC_ABC", "DOC_XYZ"],
    "Doctor_Name": ["Dr. Evans", "Dr. Evans", "Dr. Green", "Dr. Evans"],
    "Doctor_Clinic": ["General Medicine", "General Medicine", "Sports Med", "General Medicine"],
    "Prescriptions": ["Amoxicillin, Ibuprofen", "Amoxicillin, Ibuprofen", "Bandages", "Vitamin D"]
}

df_challenge = pd.DataFrame(challenge_data)

# ── Fix 1NF: explode the multi-valued Prescriptions column ──────────────────
df_1nf = df_challenge.copy()

# Split comma-separated prescriptions into a list, then explode to one row each
df_1nf["Prescriptions"] = df_1nf["Prescriptions"].str.split(", ")
df_1nf = df_1nf.explode("Prescriptions").reset_index(drop=True)

# Rename column to reflect it is now atomic
df_1nf = df_1nf.rename(columns={"Prescriptions": "Prescription"})

# Drop exact duplicate rows
df_1nf = df_1nf.drop_duplicates().reset_index(drop=True)

# Save to SQLite
df_1nf.to_sql('Patient_Visits_1NF', conn_challenge, index=False, if_exists='replace')

print("--- Normalized to 1NF ---")
print(df_1nf.to_string(index=False))

--- Normalized to 1NF ---
 Visit_ID  Student_ID Student_Name Doctor_ID Doctor_Name    Doctor_Clinic Prescription
     5001         101        Alice   DOC_XYZ   Dr. Evans General Medicine  Amoxicillin
     5001         101        Alice   DOC_XYZ   Dr. Evans General Medicine    Ibuprofen
     5002         102          Bob   DOC_ABC   Dr. Green       Sports Med     Bandages
     5003         104        David   DOC_XYZ   Dr. Evans General Medicine    Vitamin D
